**Title:** LRR Case Study - Exploratory Analysis for Part 2<br>
**Created By:** Begimay Ahmatova<br>
**Created Date:** 2026-10-02<br>
**Description:** Deep dive into raw files for company crosswalk & entity resolution.<br>

In [1]:
## libraries
import duckdb as db
import pandas as pd

In [2]:
con = db.connect("lrr.duckdb", read_only=True)

In [3]:
company = con.sql("SELECT * FROM deduped_company").df()
company.head()

,id,normalized_name,informalname,cleaned_informalname,website,domain,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date
0,9700,Medi-Code,Medi-Code,medi-code,http://medi-code.com,medi-code.com,+18324372029,Katy,Texas,77494,United States,2009,1,<NA>,NaN,<NA>,False,2026-09-15
1,20074,Marketware,Marketware,marketware,https://marketware.com,marketware.com,+18019444230,Sandy,Utah,84094,United States,2014,4,1665037,Medsphere Systems Corporation,5900000,False,2026-07-16
2,28462,VCNetwork,VCNetwork,vcnetwork,https://vcnetwork.org,vcnetwork.org,,San Francisco,California,94114,United States,1997,5,<NA>,NaN,<NA>,False,2026-09-11
3,35837,Complipoint,Complipoint,complipoint,https://www.complipoint.com,complipoint.com,+18586732174,San Diego,California,92127,United States,2002,8,204227,Applications International Corp. (AIC),<NA>,False,2025-11-26
4,45311,FlexiInternational Software,FlexiInternational Software,flexiinternational software,https://www.flexi.com,flexi.com,+18003539492,Shelton,Connecticut,06484,United States,1992,3,<NA>,NaN,<NA>,False,2026-08-08


In [4]:
accounts = pd.read_csv("account.csv")
accounts.head()

,id,isdeleted,masterrecordid,name,type,recordtypeid,parentid,billingstreet,billingcity,billingstate,...,fintrx_x3_year_aum_growth__c,fintrx_x5_year_aum_growth__c,acquired_merged__c,keystone_recommended_lead_score__c,rls_delta__c,last_logged_activity__c,acquired_by_merged_with__c,company_source__c,source__c,estimated_arr__c
0,00157Q1WAHSCDPHCUN,False,NaN,"Rare Earth Exploration, LLC",NaN,NaN,NaN,NaN,Garland,Texas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,001R7G4RI0GA09H5ZJ,False,NaN,Kenton Grant Consulting,NaN,NaN,NaN,NaN,Tulsa,Oklahoma,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0012ABVJ564CCELZ4K,False,NaN,"GUTHY-RENKER, LLC",NaN,NaN,NaN,NaN,El Segundo,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,001R0ST0DTW00BXMZZ,False,NaN,SOS TRUCK SERVICE,NaN,NaN,NaN,NaN,Ontario,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00186JM0HJK76GBGEK,False,NaN,Nutrition Partners Inc.,NaN,NaN,NaN,NaN,Airdrie,Alberta,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
## data types
pd.set_option("display.max_rows", None)
print(accounts.dtypes.to_string(), "\n")

## null / distinct profile for key columns
cols = ["id", "name", "billingcity", "billingstate", "billingcountry", "phone", "website", "domain__c"]
print("rows:", len(accounts))

profile = pd.DataFrame({
    "dtype": accounts[cols].dtypes.astype(str),
    "nulls": accounts[cols].isna().sum(),
    "null_pct": (accounts[cols].isna().mean() * 100).round(2),
    "distinct": accounts[cols].nunique(dropna=True),
})
profile["distinct_pct"] = (profile["distinct"] / (len(accounts) - profile["nulls"]) * 100).round(2)
profile

id                                               str
isdeleted                                       bool
masterrecordid                               float64
name                                             str
type                                         float64
recordtypeid                                 float64
parentid                                     float64
billingstreet                                float64
billingcity                                      str
billingstate                                     str
billingpostalcode                            float64
billingcountry                                   str
billinglatitude                              float64
billinglongitude                             float64
billinggeocodeaccuracy                       float64
billingaddress                               float64
shippingstreet                               float64
shippingcity                                 float64
shippingstate                                f

,dtype,nulls,null_pct,distinct,distinct_pct
id,str,0,0.0,100,100.0
name,str,0,0.0,100,100.0
billingcity,str,0,0.0,83,83.0
billingstate,str,0,0.0,35,35.0
billingcountry,str,0,0.0,2,2.0
phone,str,6,6.0,94,100.0
website,str,9,9.0,91,100.0
domain__c,str,4,4.0,96,100.0


In [6]:
cols = [
    "id", "name", "billingcity", "billingstate", "billingcountry",
    "phone", "website", "domain__c",
    "ss_company_name__c", "ss_website__c", "ss_city__c",
    "ss_state__c", "ss_phone__c", "ss_year_founded__c",
]

accounts_sub = accounts[cols].copy()
accounts_sub.head()

,id,name,billingcity,billingstate,billingcountry,phone,website,domain__c,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016.0
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014.0
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988.0
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967.0
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003.0


In [7]:
accounts_sub.dtypes

id                        str
name                      str
billingcity               str
billingstate              str
billingcountry            str
phone                     str
website                   str
domain__c                 str
ss_company_name__c        str
ss_website__c             str
ss_city__c                str
ss_state__c               str
ss_phone__c               str
ss_year_founded__c    float64
dtype: object

In [8]:
accounts_sub["ss_year_founded__c"] = accounts_sub["ss_year_founded__c"].astype("Int64")

In [9]:
accounts_sub = accounts_sub.rename(columns={"domain__c": "domain"})

In [10]:
pd.set_option("display.max_columns", None)   # show all columns

In [11]:
check = accounts_sub.merge(company, on="domain", how="left", indicator=True)

print(check["_merge"].value_counts())
check.head()


_merge
both          62
left_only     38
right_only     0
Name: count, dtype: int64


,id_x,name,billingcity,billingstate,billingcountry,phone,website_x,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,id_y,normalized_name,informalname,cleaned_informalname,website_y,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date,_merge
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,840010.0,Rare Earth Exploration,Rare Earth Exploration,rare earth exploration,https://www.rareearthex.com,+14694803141,Garland,Texas,75044,United States,2016,1.0,<NA>,NaN,<NA>,False,2025-04-04,both
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,21859623.0,Kenton Grant Consulting,Kenton Grant Consulting,kenton grant consulting,https://www.kentongrantconsulting.com,+19188123774,Tulsa,Oklahoma,74101,United States,2014,1.0,<NA>,NaN,<NA>,False,2026-08-11,both
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,1035025.0,Guthy-Renker,Guthy-Renker,guthy-renker,https://www.guthy-renker.com,+18886516602,El Segundo,California,90245,United States,1988,1.0,<NA>,NaN,<NA>,False,2025-09-06,both
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,708165.0,Nutrition Partners,Nutrition Partners,nutrition partners,https://nutritionpartners.ca,+14039120735,Airdrie,Alberta,T4A 2G8,Canada,2003,1.0,<NA>,NaN,<NA>,False,2025-04-19,both


In [12]:
accounts_sub.loc[accounts_sub["domain"].isna()]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967
10,001NF6AKQPMKUMYVPY,Dorado Industries,Rancho Palos Verdes,California,United States,310-544-1316,NaN,NaN,Dorado Industries,NaN,Rancho Palos Verdes,California,+13105441316,2006
30,0019JZFX6KJWSK7KEG,Full Canopy Tree Care,Austin,Texas,United States,+15123368733,NaN,NaN,"Full Canopy Tree Care, LLC",NaN,Austin,Texas,(512) 336-8733,2003
31,001XF266CCIFU6FD6Y,"CONCRETE SOFTWARE, INC.",Eden Prairie,Minnesota,United States,+19529425162,NaN,NaN,"Concrete Software, Inc.",https://concretesoftware.com,Eden Prairie,Minnesota,+19529425162,2003


In [39]:
pd.set_option("display.max_rows", 20)
accounts_sub.loc[
    accounts_sub["domain"].astype("string").str.startswith("www.", na=False)
]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std
14,0018HSSRRXQQM2PLPP,"Burke Estate & Tax Law, LLC",Murfreesboro,Tennessee,United States,+16152171414,https://burketaxlaw.com,www.burketaxlaw.com,"Burke Estate & Tax Law, LLC",https://burketaxlaw.com,Murfreesboro,Tennessee,+16152171414,2020,+16152171414,+16152171414
16,001TINX4KIAPJ2GEJR,Eve and Nico Gifts and Home Decor,Englewood,New Jersey,United States,(212) 280-0999,NaN,www.eveandnico.com,Eve and Nico Gifts and Home Decor,https://eveandnico.com,Englewood,New Jersey,212-280-0999,2014,+12122800999,+12122800999
17,0012WM4F8U7318JZFD,GADSDEN COUNTRY CLUB,Gadsden,Alabama,United States,(256) 546-0451,https://www.thegadsdencountryclub.com,www.thegadsdencountryclub.com,Gadsden Country Club,https://thegadsdencountryclub.com,Gadsden,Alabama,256-546-0451,1919,+12565460451,+12565460451
22,001KGTQ9BBGMQB37P2,Proby's Tax & Accounting Service,Los Angeles,California,United States,+13232909034,https://probystax.com,www.probystax.com,Proby's Tax & Accounting Service,https://probystax.com,Los Angeles,California,323-290-9034,1984,+13232909034,+13232909034
26,0015HVMUTIFCZ9Z8DZ,Sanskar Culture Society,Providence,Rhode Island,United States,(888) 864-9262,https://sanskarusa.org,www.sanskarusa.org,Sanskar Culture Society,https://sanskarusa.org,Foxborough,Massachusetts,888-864-9262,2016,+18888649262,+18888649262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,001ZQ3ABUUD0VKFBJN,Nixon Peabody Trust Co.,Boston,Massachusetts,United States,+16173451309,NaN,www.nixonpeabodytrustcompany.com,Nixon Peabody Trust Co.,https://nixonpeabodytrustcompany.com,Boston,Massachusetts,+16173451309,2019,+16173451309,+16173451309
92,001EVIAMR8AUBNUUB5,Sylvan Dale Guest Ranch,Loveland,Colorado,United States,970-667-3915,https://sylvandale.com,www.sylvandale.com,Sylvan Dale Guest Ranch,https://sylvandale.com,Loveland,Colorado,+19706673915,1946,+19706673915,+19706673915
94,00148AY13F2LOGQOCH,Encore Labs,Pasadena,California,United States,+16266963086,https://encorelabs.com,www.encorelabs.com,"Encore Labs, LLC",https://encorelabs.com,Pasadena,California,626-696-3086,2017,+16266963086,+16266963086
95,001EH1W9PYM3SWP1CR,"CHAMPLAIN VALLEY ORGANICS, LLC DBA VERMONT PUR...",Shoreham,Vermont,United States,(802) 282-3058,https://www.vermontpurecbd.com,www.vermontpurecbd.com,"Champlain Valley Organics, LLC dba Vermont Pur...",https://vermontpurecbd.com,Shoreham,Vermont,802-282-3058,2017,+18022823058,+18022823058


In [32]:
accounts_sub.loc[
    accounts_sub["domain"].astype("string").str.startswith("https", na=False)
]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std


In [13]:
accounts_sub.loc[accounts_sub["website"].isna()]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967
10,001NF6AKQPMKUMYVPY,Dorado Industries,Rancho Palos Verdes,California,United States,310-544-1316,NaN,NaN,Dorado Industries,NaN,Rancho Palos Verdes,California,+13105441316,2006
16,001TINX4KIAPJ2GEJR,Eve and Nico Gifts and Home Decor,Englewood,New Jersey,United States,(212) 280-0999,NaN,www.eveandnico.com,Eve and Nico Gifts and Home Decor,https://eveandnico.com,Englewood,New Jersey,212-280-0999,2014
30,0019JZFX6KJWSK7KEG,Full Canopy Tree Care,Austin,Texas,United States,+15123368733,NaN,NaN,"Full Canopy Tree Care, LLC",NaN,Austin,Texas,(512) 336-8733,2003
31,001XF266CCIFU6FD6Y,"CONCRETE SOFTWARE, INC.",Eden Prairie,Minnesota,United States,+19529425162,NaN,NaN,"Concrete Software, Inc.",https://concretesoftware.com,Eden Prairie,Minnesota,+19529425162,2003
35,001CRGEWM2YBDOZC2D,"KAUFMAN CHAMBER OF COMMERCE, INC.",Kaufman,Texas,United States,(972) 932-3118,NaN,kaufmanchamber.com,NaN,NaN,NaN,NaN,NaN,<NA>
40,0010Z2EAYJ409GF4NJ,FIKE'S ROOFING & CONSTRUCTION,Shelton,Washington,United States,360-490-0013,NaN,fikesroofing.com,NaN,https://fikesroofing.com,Shelton,Washington,+13604900013,2023
75,001NCDRTMHT2HKU23X,M247,St. Petersburg,Florida,United States,844-284-6247,NaN,www.m247.net,M247,https://m247.net,St. Petersburg,Florida,+18442846247,2018
83,001ZQ3ABUUD0VKFBJN,Nixon Peabody Trust Co.,Boston,Massachusetts,United States,+16173451309,NaN,www.nixonpeabodytrustcompany.com,Nixon Peabody Trust Co.,https://nixonpeabodytrustcompany.com,Boston,Massachusetts,+16173451309,2019


In [44]:
accounts_sub.loc[
    accounts_sub["website"].astype("string").str.startswith("www", na=False)
]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std


In [41]:
pd.set_option("display.max_rows", 20)
accounts_sub.loc[
    accounts_sub["website"].astype("string").str.startswith("https", na=False)
]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,+14694803141,+14694803141
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,+19188123774,+19188123774
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,+18886516602,+18886516602
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,+14039120735,+14039120735
5,001PMACI6O1GBDUEHH,Lucent Path Advisors,Waupun,Wisconsin,United States,(932) 385-6729,https://lucentpathadvisors.com,lucentpathadvisors.com,Lucent Path Advisors,https://lucentpathadvisors.com,Waupun,Wisconsin,(932) 385-6729,2022,+19323856729,+19323856729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,001EH1W9PYM3SWP1CR,"CHAMPLAIN VALLEY ORGANICS, LLC DBA VERMONT PUR...",Shoreham,Vermont,United States,(802) 282-3058,https://www.vermontpurecbd.com,www.vermontpurecbd.com,"Champlain Valley Organics, LLC dba Vermont Pur...",https://vermontpurecbd.com,Shoreham,Vermont,802-282-3058,2017,+18022823058,+18022823058
96,0018EYGPNNHCCFS4GI,"AMS Caregivers, Inc.",Emmaus,Pennsylvania,United States,+16109667033,https://amscaregivers.com,amscaregivers.com,"AMS Caregivers, Inc.",https://amscaregivers.com,Emmaus,Pennsylvania,+16109667033,1997,+16109667033,+16109667033
97,0012L1ITBHJAITJ6WG,Bateman Laboratories Inc.,Toronto,Ontario,Canada,(942) 573-8417,https://batemanlabs.ca,batemanlabs.ca,Bateman Laboratories Inc.,https://batemanlabs.ca,Toronto,Ontario,(942) 573-8417,1993,+19425738417,+19425738417
98,001JM05Z2V7FKXUXET,"Retirement Strategies Group, LLC (RSG)",New Orleans,Louisiana,United States,(504) 712-0005,https://www.rsgweb.com,www.rsgweb.com,"Retirement Strategies Group, LLC (RSG)",https://rsgweb.com,New Orleans,Louisiana,504-712-0005,1999,+15047120005,+15047120005


In [43]:
pd.set_option("display.max_rows", 20)
accounts_sub.loc[
    accounts_sub["ss_website__c"].astype("string").str.startswith("www.", na=False)
]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std


In [42]:
pd.set_option("display.max_rows", 20)
accounts_sub.loc[
    accounts_sub["ss_website__c"].astype("string").str.startswith("https", na=False)
]

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,+14694803141,+14694803141
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,+19188123774,+19188123774
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,+18886516602,+18886516602
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,+14039120735,+14039120735
5,001PMACI6O1GBDUEHH,Lucent Path Advisors,Waupun,Wisconsin,United States,(932) 385-6729,https://lucentpathadvisors.com,lucentpathadvisors.com,Lucent Path Advisors,https://lucentpathadvisors.com,Waupun,Wisconsin,(932) 385-6729,2022,+19323856729,+19323856729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,001EH1W9PYM3SWP1CR,"CHAMPLAIN VALLEY ORGANICS, LLC DBA VERMONT PUR...",Shoreham,Vermont,United States,(802) 282-3058,https://www.vermontpurecbd.com,www.vermontpurecbd.com,"Champlain Valley Organics, LLC dba Vermont Pur...",https://vermontpurecbd.com,Shoreham,Vermont,802-282-3058,2017,+18022823058,+18022823058
96,0018EYGPNNHCCFS4GI,"AMS Caregivers, Inc.",Emmaus,Pennsylvania,United States,+16109667033,https://amscaregivers.com,amscaregivers.com,"AMS Caregivers, Inc.",https://amscaregivers.com,Emmaus,Pennsylvania,+16109667033,1997,+16109667033,+16109667033
97,0012L1ITBHJAITJ6WG,Bateman Laboratories Inc.,Toronto,Ontario,Canada,(942) 573-8417,https://batemanlabs.ca,batemanlabs.ca,Bateman Laboratories Inc.,https://batemanlabs.ca,Toronto,Ontario,(942) 573-8417,1993,+19425738417,+19425738417
98,001JM05Z2V7FKXUXET,"Retirement Strategies Group, LLC (RSG)",New Orleans,Louisiana,United States,(504) 712-0005,https://www.rsgweb.com,www.rsgweb.com,"Retirement Strategies Group, LLC (RSG)",https://rsgweb.com,New Orleans,Louisiana,504-712-0005,1999,+15047120005,+15047120005


In [14]:
accounts_sub[["phone", "ss_phone__c"]].isna().sum()

phone           6
ss_phone__c    21
dtype: int64

In [15]:
digits_phone = accounts_sub["phone"].astype("string").str.replace(r"\D", "", regex=True)
digits_phone.str.len().value_counts(dropna=False).sort_index()

phone
10      68
11      26
<NA>     6
Name: count, dtype: Int64

In [27]:
check = accounts_sub.merge(
    company,
    left_on="phone",
    right_on="normalized_phone",
    how="left",
    indicator=True,
)
print(check["_merge"].value_counts())
check.head()


_merge
left_only     74
both          26
right_only     0
Name: count, dtype: int64


,id_x,name,billingcity,billingstate,billingcountry,phone,website_x,domain_x,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std,id_y,normalized_name,informalname,cleaned_informalname,website_y,domain_y,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date,_merge
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,+14694803141,+14694803141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,+19188123774,+19188123774,21859623.0,Kenton Grant Consulting,Kenton Grant Consulting,kenton grant consulting,https://www.kentongrantconsulting.com,kentongrantconsulting.com,+19188123774,Tulsa,Oklahoma,74101,United States,2014,1.0,<NA>,NaN,<NA>,False,2026-08-11,both
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,+18886516602,+18886516602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967,+18889999767,+18889999767,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,+14039120735,+14039120735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only


In [16]:
accounts_sub["phone"].dropna().astype(str).str.replace(r"\d", "D", regex=True).value_counts()

phone
(DDD) DDD-DDDD    48
+DDDDDDDDDDD      26
DDD-DDD-DDDD      20
Name: count, dtype: int64

In [17]:
digits = accounts_sub["phone"].astype("string").str.replace(r"\D", "", regex=True)

na_countries = {"united states", "canada"}
is_na = accounts_sub["billingcountry"].astype("string").str.strip().str.lower().isin(na_countries)

m10 = (digits.str.len() == 10) & is_na
m11 = (digits.str.len() == 11) & digits.str.startswith("1")

accounts_sub["phone_std"] = accounts_sub["phone"]
accounts_sub.loc[m10, "phone_std"] = "+1" + digits[m10]
accounts_sub.loc[m11, "phone_std"] = "+" + digits[m11]

print(m10.sum(), "+1 added;", m11.sum(), "already had the 1")


68 +1 added; 26 already had the 1


In [18]:
accounts_sub["phone_std"].dropna().astype(str).str.replace(r"\d", "D", regex=True).value_counts()

phone_std
+DDDDDDDDDDD    94
Name: count, dtype: int64

In [28]:
check = accounts_sub.merge(
    company,
    left_on="phone_std",
    right_on="normalized_phone",
    how="left",
    indicator=True,
)
print(check["_merge"].value_counts())
check.head()


_merge
both          83
left_only     17
right_only     0
Name: count, dtype: int64


,id_x,name,billingcity,billingstate,billingcountry,phone,website_x,domain_x,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std,id_y,normalized_name,informalname,cleaned_informalname,website_y,domain_y,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date,_merge
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,+14694803141,+14694803141,840010.0,Rare Earth Exploration,Rare Earth Exploration,rare earth exploration,https://www.rareearthex.com,rareearthex.com,+14694803141,Garland,Texas,75044,United States,2016,1.0,<NA>,NaN,<NA>,False,2025-04-04,both
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,+19188123774,+19188123774,21859623.0,Kenton Grant Consulting,Kenton Grant Consulting,kenton grant consulting,https://www.kentongrantconsulting.com,kentongrantconsulting.com,+19188123774,Tulsa,Oklahoma,74101,United States,2014,1.0,<NA>,NaN,<NA>,False,2026-08-11,both
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,+18886516602,+18886516602,1035025.0,Guthy-Renker,Guthy-Renker,guthy-renker,https://www.guthy-renker.com,guthy-renker.com,+18886516602,El Segundo,California,90245,United States,1988,1.0,<NA>,NaN,<NA>,False,2025-09-06,both
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967,+18889999767,+18889999767,17382874.0,SOS TRUCK SERVICE,SOS TRUCK SERVICE,sos truck service,http://www.sostruckservice.com,sostruckservice.com,+18889999767,Ontario,California,91761,United States,1967,1.0,<NA>,NaN,<NA>,False,2026-02-27,both
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,+14039120735,+14039120735,708165.0,Nutrition Partners,Nutrition Partners,nutrition partners,https://nutritionpartners.ca,nutritionpartners.ca,+14039120735,Airdrie,Alberta,T4A 2G8,Canada,2003,1.0,<NA>,NaN,<NA>,False,2025-04-19,both


In [19]:
check = accounts_sub.merge(
    company,
    left_on="ss_phone__c",
    right_on="normalized_phone",
    how="left",
    indicator=True,
)
print(check["_merge"].value_counts())
check.head()


_merge
left_only     58
both          42
right_only     0
Name: count, dtype: int64


,id_x,name,billingcity,billingstate,billingcountry,phone,website_x,domain_x,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,id_y,normalized_name,informalname,cleaned_informalname,website_y,domain_y,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date,_merge
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,+14694803141,840010.0,Rare Earth Exploration,Rare Earth Exploration,rare earth exploration,https://www.rareearthex.com,rareearthex.com,+14694803141,Garland,Texas,75044,United States,2016,1.0,<NA>,NaN,<NA>,False,2025-04-04,both
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,+19188123774,21859623.0,Kenton Grant Consulting,Kenton Grant Consulting,kenton grant consulting,https://www.kentongrantconsulting.com,kentongrantconsulting.com,+19188123774,Tulsa,Oklahoma,74101,United States,2014,1.0,<NA>,NaN,<NA>,False,2026-08-11,both
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,+18886516602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967,+18889999767,17382874.0,SOS TRUCK SERVICE,SOS TRUCK SERVICE,sos truck service,http://www.sostruckservice.com,sostruckservice.com,+18889999767,Ontario,California,91761,United States,1967,1.0,<NA>,NaN,<NA>,False,2026-02-27,both
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,+14039120735,708165.0,Nutrition Partners,Nutrition Partners,nutrition partners,https://nutritionpartners.ca,nutritionpartners.ca,+14039120735,Airdrie,Alberta,T4A 2G8,Canada,2003,1.0,<NA>,NaN,<NA>,False,2025-04-19,both


In [20]:
digits = accounts_sub["ss_phone__c"].astype("string").str.replace(r"\D", "", regex=True)
digits.str.len().value_counts(dropna=False).sort_index()


ss_phone__c
10      36
11      43
<NA>    21
Name: count, dtype: Int64

In [21]:
# distinct values with record counts (includes nulls)
accounts_sub["billingcountry"].value_counts(dropna=False)


billingcountry
United States    90
Canada           10
Name: count, dtype: int64

In [22]:
accounts["ss_phone__c"].dropna().astype(str).str.replace(r"\d", "D", regex=True).value_counts()


ss_phone__c
+DDDDDDDDDDD      43
DDD-DDD-DDDD      24
(DDD) DDD-DDDD    11
DDD.DDD.DDDD       1
Name: count, dtype: int64

In [23]:
digits = accounts_sub["ss_phone__c"].astype("string").str.replace(r"\D", "", regex=True)

na_countries = {"united states", "canada"}
is_na = accounts_sub["billingcountry"].astype("string").str.strip().str.lower().isin(na_countries)

m10 = (digits.str.len() == 10) & is_na
m11 = (digits.str.len() == 11) & digits.str.startswith("1")

accounts_sub["ss_phone_std"] = accounts_sub["ss_phone__c"]
accounts_sub.loc[m10, "ss_phone_std"] = "+1" + digits[m10]
accounts_sub.loc[m11, "ss_phone_std"] = "+" + digits[m11]

print(m10.sum(), "+1 added;", m11.sum(), "already had the 1")


36 +1 added; 43 already had the 1


In [24]:
accounts_sub["ss_phone_std"].dropna().astype(str).str.replace(r"\d", "D", regex=True).value_counts()

ss_phone_std
+DDDDDDDDDDD    79
Name: count, dtype: int64

In [25]:
check = accounts_sub.merge(
    company,
    left_on="ss_phone_std",       # not ss_phone__c
    right_on="normalized_phone",
    how="left",
    indicator=True,
)
print(check["_merge"].value_counts())
check.head()


_merge
both          68
left_only     32
right_only     0
Name: count, dtype: int64


,id_x,name,billingcity,billingstate,billingcountry,phone,website_x,domain_x,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std,id_y,normalized_name,informalname,cleaned_informalname,website_y,domain_y,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date,_merge
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016,+14694803141,+14694803141,840010.0,Rare Earth Exploration,Rare Earth Exploration,rare earth exploration,https://www.rareearthex.com,rareearthex.com,+14694803141,Garland,Texas,75044,United States,2016,1.0,<NA>,NaN,<NA>,False,2025-04-04,both
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014,+19188123774,+19188123774,21859623.0,Kenton Grant Consulting,Kenton Grant Consulting,kenton grant consulting,https://www.kentongrantconsulting.com,kentongrantconsulting.com,+19188123774,Tulsa,Oklahoma,74101,United States,2014,1.0,<NA>,NaN,<NA>,False,2026-08-11,both
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988,+18886516602,+18886516602,1035025.0,Guthy-Renker,Guthy-Renker,guthy-renker,https://www.guthy-renker.com,guthy-renker.com,+18886516602,El Segundo,California,90245,United States,1988,1.0,<NA>,NaN,<NA>,False,2025-09-06,both
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967,+18889999767,+18889999767,17382874.0,SOS TRUCK SERVICE,SOS TRUCK SERVICE,sos truck service,http://www.sostruckservice.com,sostruckservice.com,+18889999767,Ontario,California,91761,United States,1967,1.0,<NA>,NaN,<NA>,False,2026-02-27,both
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003,+14039120735,+14039120735,708165.0,Nutrition Partners,Nutrition Partners,nutrition partners,https://nutritionpartners.ca,nutritionpartners.ca,+14039120735,Airdrie,Alberta,T4A 2G8,Canada,2003,1.0,<NA>,NaN,<NA>,False,2025-04-19,both


In [26]:
check[check["_merge"] == "left_only"]


,id_x,name,billingcity,billingstate,billingcountry,phone,website_x,domain_x,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,phone_std,ss_phone_std,id_y,normalized_name,informalname,cleaned_informalname,website_y,domain_y,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date,_merge
5,001PMACI6O1GBDUEHH,Lucent Path Advisors,Waupun,Wisconsin,United States,(932) 385-6729,https://lucentpathadvisors.com,lucentpathadvisors.com,Lucent Path Advisors,https://lucentpathadvisors.com,Waupun,Wisconsin,(932) 385-6729,2022,+19323856729,+19323856729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
7,001VOQ4CT939RX77RI,Happy Period,Palmdale,California,United States,NaN,https://hashtaghappyperiod.org,hashtaghappyperiod.org,Happy Period,https://hashtaghappyperiod.org,Palmdale,California,NaN,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
9,0017RUE8OQQ4W74OJE,Quarrystone Capital Partners,Cary,North Carolina,United States,(925) 354-2292,https://quarrystonecapital.com,quarrystonecapital.com,Quarrystone Capital Partners,https://quarrystonecapital.com,Cary,North Carolina,(925) 354-2292,1992,+19253542292,+19253542292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
11,001DJZDNBJ0DDLZ2UH,PrecisionChem LLC,Waupun,Wisconsin,United States,+19203242007,https://precisionchemllc.com,precisionchemllc.com,NaN,NaN,NaN,NaN,NaN,<NA>,+19203242007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
12,001UAY5GCQ8NKM7WG3,Helix Intelligence Group,San Francisco,California,United States,NaN,https://www.centrum-ai.com,centrum-ai.com,Centrum-AI Inc.,https://centrum-ai.com,San Francisco,California,NaN,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
13,001LWUOXI9XQPDCGZD,OpenText Enterprise Apps Fund (OTEAF),Toronto,Ontario,Canada,416-306-3020,https://firstascenntures.com,firstascenntures.com,NaN,NaN,NaN,NaN,NaN,<NA>,+14163063020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
15,001OTNZEKJCBHGKWJB,Professional Communicators LLC,Littleton,Colorado,United States,(866) 785-8255,https://procommunicators.com,procommunicators.com,NaN,NaN,NaN,NaN,NaN,<NA>,+18667858255,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
18,0014B0PZCYC3EDQMEV,"Overhead Door of Nova Scotia, Ltd",Dartmouth,Nova Scotia,Canada,(902) 455-5469,https://ohdns.ca,ohdns.ca,NaN,NaN,NaN,NaN,NaN,<NA>,+19024555469,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
27,001RT5NK4RITSFVA5P,Palmwood Dental Partners,Rancho Palos Verdes,California,United States,(542) 661-7381,https://palmwooddental.com,palmwooddental.com,Palmwood Dental Partners,https://palmwooddental.com,Rancho Palos Verdes,California,(542) 661-7381,2005,+15426617381,+15426617381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only
29,001BJTAYFLOUMGE9X6,Thirteen Barrels Brewing Co.,Lafayette,Colorado,United States,(303) 997-4164,https://odd13brewing.com,odd13brewing.com,NaN,NaN,NaN,NaN,NaN,<NA>,+13039974164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaT,left_only


- 4 recs dont have domains in accounts
- 9 recs dont have websites
- standardizing ss_phone__c improves match from 42 to 68 (+26), 21/32 still null (11 genuinely have no match)
- standardizing phone improves match from 26 to 83 (+57), 6/17 still null (11 genuinely have no match)
- ss_year_founded__c convert to int

In [29]:
con.close()